In [ ]:
%cd ..

In [ ]:
import os
import glob
import pandas as pd

from state_predictor.coder import Coder
from src.model.yolo_handler import YoloHandler
from utils.config_parser import ConfigParser
from utils.common_utils import normalize, denormalize

dataset = "kitti"
model_conf = ConfigParser.read(f"config/model/{dataset}_model.ini")


In [ ]:
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np


def compare_layerwise_pruning(initial_model, pruned_models, model_labels=None):
    """
    Bar-plot pruning fraction per Conv2d layer for multiple pruned models.

    initial_model: baseline model (unpruned)
    pruned_models: list of pruned models
    model_labels: list of names for legend
    """

    if not isinstance(pruned_models, (list, tuple)):
        pruned_models = [pruned_models]

    if model_labels is None:
        model_labels = [f"model_{i}" for i in range(len(pruned_models))]

    # conv layers in baseline
    init_convs = [(n, m) for n, m in initial_model.named_modules()
                  if isinstance(m, nn.Conv2d)]
    layer_names = [n for n, _ in init_convs]

    # collect pruning fractions for each model
    all_fracs = []

    for pm in pruned_models:
        pm_convs = {n: m for n, m in pm.named_modules()
                    if isinstance(m, nn.Conv2d)}

        fracs = []
        for name, conv0 in init_convs:
            conv1 = pm_convs.get(name, None)
            if conv1 is None:
                fracs.append(np.nan)
                continue
            frac = 1.0 - (conv1.out_channels / conv0.out_channels)
            fracs.append(frac)

        all_fracs.append(fracs)

    all_fracs = np.array(all_fracs)  # (n_models, n_layers)

    # ---- Plot grouped bars ----
    n_models, n_layers = all_fracs.shape
    x = np.arange(n_layers)
    width = 0.8 / n_models

    plt.figure(figsize=(max(12, 0.35 * n_layers), 6))

    for i in range(n_models):
        plt.bar(x + i * width, all_fracs[i], width, label=model_labels[i])

    plt.xticks(x + width * (n_models - 1) / 2, layer_names, rotation=90)
    plt.ylabel("Pruned fraction (out_channels)")
    plt.title("Layer-wise pruning comparison")
    plt.legend()
    plt.tight_layout()
    plt.show()

    return layer_names, all_fracs


In [ ]:
yolo_handler_lamp = YoloHandler(model_conf.model)
yolo_handler_lamp.load_pruned_model(path=f"runs/pruned/pruned_lamp_15.pt")

yolo_handler_random = YoloHandler(model_conf.model)
yolo_handler_random.load_pruned_model(path=f"runs/pruned/pruned_ours_26.pt")

compare_layerwise_pruning(
    yolo_handler_lamp._init_model,
    [yolo_handler_lamp._model, yolo_handler_random._model],
    model_labels=["lamp", "ours"]
)


In [ ]:
yolo_handler_lamp = YoloHandler(model_conf.model)
yolo_handler_lamp.load_pruned_model(path=f"runs/pruned/pruned_lamp_30.pt")

yolo_handler_random = YoloHandler(model_conf.model)
yolo_handler_random.load_pruned_model(path=f"runs/pruned/pruned_ours_43.pt")

compare_layerwise_pruning(
    yolo_handler_lamp._init_model,
    [yolo_handler_lamp._model, yolo_handler_random._model],
    model_labels=["lamp", "ours"]
)


In [ ]:
yolo_handler_lamp = YoloHandler(model_conf.model)
yolo_handler_lamp.load_pruned_model(path=f"runs/pruned/pruned_lamp_15.pt")

yolo_handler_random = YoloHandler(model_conf.model)
yolo_handler_random.load_pruned_model(path=f"runs/pruned/pruned_random_20.pt")

compare_layerwise_pruning(
    yolo_handler_lamp._init_model,
    [yolo_handler_lamp._model, yolo_handler_random._model],
    model_labels=["lamp", "ours"]
)
